# DeepPulse — LSTM RUL Prediction on NASA C-MAPSS

This notebook trains a 2-layer LSTM to predict Remaining Useful Life (RUL) of turbofan engines from the NASA C-MAPSS FD001 dataset.

**Runtime:** GPU (T4 recommended, free on Colab)  
**Dataset:** NASA C-MAPSS — download link in cell 2

## 1. Install dependencies

In [ ]:
!pip install tensorflow pandas numpy scikit-learn matplotlib -q

## 2. Load dataset

In [ ]:
import pandas as pd
import numpy as np

# Download NASA C-MAPSS if not already present
import os
if not os.path.exists("CMAPSSData"):
    !wget -q https://ti.arc.nasa.gov/c/6/ -O cmapss.zip
    !unzip -q cmapss.zip -d CMAPSSData

COLS = ["unit", "cycle", "set1", "set2", "set3"] + [f"s{i}" for i in range(1, 22)]
FLAT = {"s1", "s5", "s6", "s10", "s16", "s18", "s19"}
SENSORS = [s for s in [f"s{i}" for i in range(1, 22)] if s not in FLAT]
RUL_CAP = 125

train = pd.read_csv("CMAPSSData/train_FD001.txt", sep=r"\s+", header=None, names=COLS)
test  = pd.read_csv("CMAPSSData/test_FD001.txt",  sep=r"\s+", header=None, names=COLS)
rul   = pd.read_csv("CMAPSSData/RUL_FD001.txt",   header=None, names=["final_rul"])
rul["unit"] = range(1, len(rul) + 1)

# Compute piecewise-capped RUL for training
max_cycles = train.groupby("unit")["cycle"].transform("max")
train["actual_rul"] = (max_cycles - train["cycle"]).clip(upper=RUL_CAP)

print(f"Train: {len(train)} rows, {train['unit'].nunique()} engines")
print(f"Test:  {len(test)} rows, {test['unit'].nunique()} engines")
print(f"Using {len(SENSORS)} sensors: {SENSORS}")

## 3. Normalize sensors

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train[SENSORS] = scaler.fit_transform(train[SENSORS])
test[SENSORS]  = scaler.transform(test[SENSORS])

print("Normalized. Sample stats:")
train[SENSORS].describe().round(3)

## 4. Create sliding window sequences

In [ ]:
SEQ_LEN = 30

def make_sequences(df, seq_len, sensors, target_col="actual_rul"):
    X, y = [], []
    for unit in df["unit"].unique():
        u = df[df["unit"] == unit].reset_index(drop=True)
        for i in range(len(u) - seq_len + 1):
            X.append(u[sensors].iloc[i:i+seq_len].values)
            y.append(u[target_col].iloc[i+seq_len-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_train, y_train = make_sequences(train, SEQ_LEN, SENSORS)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

# For test, use last SEQ_LEN cycles per engine + true RUL from RUL file
def make_test_sequences(df, seq_len, sensors, rul_df):
    X, y = [], []
    for unit in df["unit"].unique():
        u = df[df["unit"] == unit].reset_index(drop=True)
        final_rul = int(rul_df[rul_df["unit"] == unit]["final_rul"].values[0])
        seq = u[sensors].iloc[-seq_len:].values
        if len(seq) < seq_len:
            pad = np.zeros((seq_len - len(seq), len(sensors)))
            seq = np.vstack([pad, seq])
        X.append(seq)
        y.append(min(RUL_CAP, final_rul))
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_test, y_test = make_test_sequences(test, SEQ_LEN, SENSORS, rul)
print(f"X_test:  {X_test.shape}, y_test: {y_test.shape}")

## 5. Build and train LSTM

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, callbacks

tf.random.set_seed(42)

model = tf.keras.Sequential([
    layers.Input(shape=(SEQ_LEN, len(SENSORS))),
    layers.LSTM(128, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(64),
    layers.Dropout(0.2),
    layers.Dense(1),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="mse",
    metrics=["mae"],
)
model.summary()

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    callbacks=[
        callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(patience=4, factor=0.5),
    ],
    verbose=1,
)

## 6. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["loss"],     label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("MSE Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["mae"],     label="Train MAE")
axes[1].plot(history.history["val_mae"], label="Val MAE")
axes[1].set_title("MAE (cycles)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on test set

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

y_pred = model.predict(X_test, verbose=0).flatten()
y_pred = np.clip(y_pred, 0, 130)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:.2f} cycles")
print(f"Test MAE:  {mae:.2f} cycles")
print(f"Test R²:   {r2:.4f}")

# Scatter plot: actual vs predicted
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.6, color="#0ea5e9")
plt.plot([0, 130], [0, 130], "--", color="gray", label="Perfect")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title(f"LSTM Predictions — RMSE={rmse:.1f}")
plt.legend()
plt.show()

## 8. Export pre-computed predictions for the dashboard

In [ ]:
import json

# Save per-cycle predictions for all test engines
records = []
for unit in test["unit"].unique():
    u = test[test["unit"] == unit].reset_index(drop=True)
    final_rul = int(rul[rul["unit"] == unit]["final_rul"].values[0])
    max_c = int(u["cycle"].max())
    for _, row in u.iterrows():
        c = int(row["cycle"])
        remaining = max_c - c
        actual_r = min(RUL_CAP, final_rul + remaining)
        records.append({"unit": unit, "cycle": c, "actual_rul": actual_r})

pred_df = pd.DataFrame(records)

# Get predictions for each window
X_all, idx_all = [], []
for unit in test["unit"].unique():
    u = test[test["unit"] == unit].reset_index(drop=True)
    for i in range(len(u)):
        start = max(0, i - SEQ_LEN + 1)
        seq = u[SENSORS].iloc[start:i+1].values
        if len(seq) < SEQ_LEN:
            pad = np.zeros((SEQ_LEN - len(seq), len(SENSORS)))
            seq = np.vstack([pad, seq])
        X_all.append(seq)
        idx_all.append((unit, int(u["cycle"].iloc[i])))

X_all = np.array(X_all, dtype=np.float32)
preds_all = model.predict(X_all, batch_size=512, verbose=0).flatten()
preds_all = np.clip(preds_all, 0, 130)

pred_map = {(u, c): float(round(p, 1)) for (u, c), p in zip(idx_all, preds_all)}
pred_df["predicted_rul"] = pred_df.apply(lambda r: pred_map.get((r["unit"], r["cycle"]), 0.0), axis=1)

pred_df.to_csv("lstm_predictions_FD001.csv", index=False)
print(f"Saved {len(pred_df)} rows to lstm_predictions_FD001.csv")

# Save training history
h = history.history
hist_out = {
    "epoch": list(range(1, len(h["loss"]) + 1)),
    "train_loss": [round(float(v), 5) for v in h["loss"]],
    "val_loss":   [round(float(v), 5) for v in h["val_loss"]],
}
with open("training_history.json", "w") as f:
    json.dump(hist_out, f, indent=2)
print("Saved training_history.json")

# Save evaluation metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
naive_rmse = float(np.sqrt(mean_squared_error(y_test, np.full_like(y_test, y_test.mean()))))
eval_meta = {
    "model": "LSTM (2-layer, 128 units)",
    "dataset": "NASA C-MAPSS FD001",
    "test_engines": 100,
    "rmse": round(float(rmse), 2),
    "mae": round(float(mae), 2),
    "r2": round(float(r2), 4),
    "baseline_rmse": round(naive_rmse, 2),
    "improvement_vs_baseline_pct": round((naive_rmse - float(rmse)) / naive_rmse * 100, 1),
    "training_epochs": len(h["loss"]),
    "sequence_length": SEQ_LEN,
    "features_used": len(SENSORS),
    "rul_cap": RUL_CAP,
}
with open("model_evaluation.json", "w") as f:
    json.dump(eval_meta, f, indent=2)
print("Saved model_evaluation.json")